In [2]:
import numpy as np
import pandas as pd
import json 
import ast

In [3]:
df = pd.read_csv('SGJobData.CSV')
print(df.shape)

(1048585, 22)


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVac

In [5]:
df.iloc[0:50000].to_excel('first 50k.xlsx')

In [6]:
df.columns

Index(['categories', 'employmentTypes', 'metadata_expiryDate',
       'metadata_isPostedOnBehalf', 'metadata_jobPostId',
       'metadata_newPostingDate', 'metadata_originalPostingDate',
       'metadata_repostCount', 'metadata_totalNumberJobApplication',
       'metadata_totalNumberOfView', 'minimumYearsExperience',
       'numberOfVacancies', 'occupationId', 'positionLevels',
       'postedCompany_name', 'salary_maximum', 'salary_minimum', 'salary_type',
       'status_id', 'status_jobStatus', 'title', 'average_salary'],
      dtype='object')

# Cleaning up steps #
1) Change to the appropriate data type
2) Remove truly blank roles
3) Remove duplicates if any.
4) 

In [7]:
# Clean employmentTypes: drop null/blank rows, then inspect unique values
df['employmentTypes'] = df['employmentTypes'].astype('string').str.strip()
df = df[df['employmentTypes'].notna() & (df['employmentTypes'] != '')]

unique_employment_types = df['employmentTypes'].unique()
print(unique_employment_types)

<StringArray>
[            'Permanent',             'Full Time',             'Part Time',
              'Contract',             'Freelance',             'Temporary',
            'Flexi-work', 'Internship/Attachment']
Length: 8, dtype: string


In [8]:
df['employmentTypes'].value_counts(dropna=False)

Permanent                458139
Full Time                393352
Contract                 139182
Part Time                 25431
Temporary                 18241
Internship/Attachment      6959
Freelance                  2139
Flexi-work                 1154
Name: employmentTypes, dtype: Int64

In [9]:
df[df['employmentTypes'].isna() | (df['employmentTypes'].str.strip() == '')]

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary


In [10]:
(df['employmentTypes'].isna() | (df['employmentTypes'].str.strip() == '')).sum()

0

In [11]:
df['metadata_totalNumberJobApplication'].value_counts(dropna=False)

0      656375
1      246159
2       23284
3       16627
4       12949
        ...  
242         1
366         1
334         1
374         1
632         1
Name: metadata_totalNumberJobApplication, Length: 376, dtype: int64

In [12]:
categories_blank = df['categories'].isna() | (df['categories'].str.strip() == '')
employmentTypes_blank = df['employmentTypes'].isna() | (df['employmentTypes'].str.strip() == '')

print("categories blank count:", categories_blank.sum())
print("employmentTypes blank count:", employmentTypes_blank.sum())
print("both blank:", (categories_blank & employmentTypes_blank).sum())
print("categories blank but employmentTypes not:", (categories_blank & ~employmentTypes_blank).sum())
print("employmentTypes blank but categories not:", (~categories_blank & employmentTypes_blank).sum())
print("same rows (masks identical):", categories_blank.equals(employmentTypes_blank))


categories blank count: 0
employmentTypes blank count: 0
both blank: 0
categories blank but employmentTypes not: 0
employmentTypes blank but categories not: 0
same rows (masks identical): False


# Remove empty rows # (df1)

In [13]:
df1 = df[df['categories'].notna() & (df['categories'].str.strip() != '')].drop(columns=['occupationId'])
print(df1.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1044597 entries, 0 to 1048584
Data columns (total 21 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  string 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1044597 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1044597 non-null  int64  
 8   metadata_totalNumberJobApplication  1044597 non-null  int64  
 9   metadata_totalNumberOfView          1044597 non-null  int64  
 10  minimumYearsExperience              1044597 non-null  int64  
 11  numberOfVac

# Check for duplicates (if any) - but first change data type to the right ones #

In [14]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1044597 entries, 0 to 1048584
Data columns (total 21 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  string 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1044597 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1044597 non-null  int64  
 8   metadata_totalNumberJobApplication  1044597 non-null  int64  
 9   metadata_totalNumberOfView          1044597 non-null  int64  
 10  minimumYearsExperience              1044597 non-null  int64  
 11  numberOfVac

In [15]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1044597 entries, 0 to 1048584
Data columns (total 21 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  string 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1044597 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1044597 non-null  int64  
 8   metadata_totalNumberJobApplication  1044597 non-null  int64  
 9   metadata_totalNumberOfView          1044597 non-null  int64  
 10  minimumYearsExperience              1044597 non-null  int64  
 11  numberOfVac

In [16]:
print("duplicate jobPostIds:", df1['metadata_jobPostId'].duplicated().sum())
df1[df1['metadata_jobPostId'].duplicated(keep=False)].sort_values('metadata_jobPostId')

duplicate jobPostIds: 0


,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,numberOfVacancies,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary


In [17]:
import json

df1['categories_parsed'] = df1['categories'].apply(json.loads)
df1['categories_parsed'].head().to_excel('categories_parsed check.xlsx')

In [18]:
df_categories = df1[['metadata_jobPostId', 'categories_parsed']].explode('categories_parsed')
df_categories = df_categories.dropna(subset=['categories_parsed'])
df_categories['category_id'] = df_categories['categories_parsed'].apply(lambda d: d['id'])
df_categories['category'] = df_categories['categories_parsed'].apply(lambda d: d['category'])
df_categories = df_categories.drop(columns='categories_parsed')

df_categories.head()

,metadata_jobPostId,category_id,category
0,MCF-2023-0252866,13,Environment / Health
0,MCF-2023-0252866,25,Manufacturing
0,MCF-2023-0252866,36,Sciences / Laboratory / R&D
1,MCF-2023-0273977,21,Information Technology
2,MCF-2023-0273994,33,Repair and Maintenance


In [19]:
def parse_json(val):
    if isinstance(val, str):
        return json.loads(val)
    return val

In [20]:
records = df1['categories'].apply(parse_json).explode()
result_df = pd.json_normalize(records)[['id', 'category']]
result_df = result_df.drop_duplicates(subset='id').reset_index(drop=True)
result_df.head()


,id,category
0,13,Environment / Health
1,25,Manufacturing
2,36,Sciences / Laboratory / R&D
3,21,Information Technology
4,33,Repair and Maintenance


In [21]:
result_df.to_excel('categories metadata.xlsx')

In [22]:
result_df = result_df.drop_duplicates(subset='id').reset_index(drop=True)
result_df

,id,category
0,13,Environment / Health
1,25,Manufacturing
2,36,Sciences / Laboratory / R&D
3,21,Information Technology
4,33,Repair and Maintenance
5,2,Admin / Secretarial
6,7,Consulting
7,29,Professional Services
8,37,Security and Investigation
9,1,Accounting / Auditing / Taxation


In [23]:
len(result_df)

43

In [24]:
result_df.to_excel('categories meta.xlsx')

In [25]:
if isinstance(df1['categories'].iloc[0], str):
    df1['categories'] = df1['categories'].apply(ast.literal_eval)

# Step 2: explode so each category dict gets its own row
df_map = df1[['metadata_jobPostId', 'categories']].explode('categories').reset_index(drop=True)

# Step 3: pull out the 'id' (and optionally 'category' name) from each dict
df_map['category_id'] = df_map['categories'].apply(lambda x: x['id'])
df_map['category_name'] = df_map['categories'].apply(lambda x: x['category'])

# Step 4: drop the now-unneeded dict column
df_map = df_map.drop(columns=['categories'])
df_map

,metadata_jobPostId,category_id,category_name
0,MCF-2023-0252866,13,Environment / Health
1,MCF-2023-0252866,25,Manufacturing
2,MCF-2023-0252866,36,Sciences / Laboratory / R&D
3,MCF-2023-0273977,21,Information Technology
4,MCF-2023-0273994,33,Repair and Maintenance
...,...,...,...
1767824,RANDOM_JOB_20251115011349636339_9,2,Admin / Secretarial
1767825,RANDOM_JOB_20251115011349636339_9,8,Customer Service
1767826,RANDOM_JOB_20251115011349636339_9,19,Hospitality
1767827,RANDOM_JOB_20251115011349636339_9,27,Medical / Therapy Services


In [26]:
df1[df1['metadata_jobPostId'] == "RANDOM_JOB_20251115011349636339_9"]

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary,categories_parsed
1048584,"[{'id': 2, 'category': 'Admin / Secretarial'},...",Permanent,2024-04-21,False,RANDOM_JOB_20251115011349636339_9,2023-11-18,2023-10-28,2,632,3920,...,Non-executive,THALES DIS (SINGAPORE) PTE. LTD.,14420727,164428,Monthly,0,Re-open,sales and operations manager,7292577.5,"[{'id': 2, 'category': 'Admin / Secretarial'},..."


In [27]:
14420727/1e6

14.420727

In [28]:
df1['positionLevels'].value_counts(dropna=False)

Executive            253701
Junior Executive     167656
Non-executive        131608
Fresh/entry level    118661
Professional         112208
Manager              110122
Senior Executive     100459
Middle Management     27375
Senior Management     22807
Name: positionLevels, dtype: int64